# Chapter 4 — 주요 공격 유형 정밀 분석 (실습)

## 실습 목표
1. CICIDS2017 구조를 모사한 합성 데이터를 생성하고 라벨 분포를 본다.
2. DDoS·PortScan·BENIGN 라벨별 주요 feature 분포 차이를 시각화한다.
3. 본문의 '스캔 의심' 규칙을 코드로 구현하고 실제 라벨과 매칭률을 평가한다.
4. 결과를 표·히트맵으로 정리한다.

## 데이터셋
본 실습은 CICIDS2017 컬럼 구조를 모사한 **교육용 합성 데이터**를 생성한다(재현성·오프라인 실행).
실제 CICIDS2017 데이터셋은 UNB 에서 받아 교체할 수 있다: <https://www.unb.ca/cic/datasets/ids-2017.html>

---
## Step 0. 환경 설정

본 실습에서는 CICIDS2017 구조를 모사한 교육용 합성 데이터를 생성한다 (실제 데이터셋은 UNB 에서 받아 교체 가능).

In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [ ]:
# === 한글 폰트 설정 (matplotlib) ===
from pathlib import Path
import matplotlib
from matplotlib import font_manager
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf']:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        matplotlib.rcParams['font.family'] = font_manager.FontProperties(fname=fp).get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q pandas
# !pip install -q numpy
# !pip install -q matplotlib

In [ ]:
# 교육용 합성 데이터셋 (CICIDS2017 컬럼 구조를 모사)
# - 외부 다운로드 없음. 시드 고정으로 결과 재현 가능
# - Colab / 로컬(Windows·macOS·Linux) 어디서나 동일하게 동작
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path.cwd() / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── 실습 과제 ──
# 목표: Step 0. 환경 설정 — 본 실습에서는 CICIDS2017 구조를 모사한 교육용 합성 데이터를 생성한다 (실제 데이터셋은 UNB 에서 받아 교체 가능).
# 만들 결과: rng, make_rows(), df
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
rng = ...
n_benign, n_ddos, n_scan = ...

def make_rows(n, label, flow_duration_range, pkts_range, syn_ratio_range, dst_port_choices):
    return ...
        'Source IP': rng.choice([f'10.0.0.{i}' for i in range(1, 50)], size = ...
        'Destination Port': rng.choice(dst_port_choices, size = ...
        ...

df = ...
    ...
], ignore_index = ...

...
```
</details>


---
## Step 1. 라벨 분포 확인

In [ ]:
# ── 실습 과제 ──
# 목표: Step 1. 라벨 분포 확인
# 만들 결과: label_pct
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
label_pct = ...
...
```
</details>


관찰: BENIGN 이 80% 이상으로 극심한 클래스 불균형. ch09 에서 이 문제를 다룬다.

---
## Step 2. 라벨별 Feature 분포 비교

In [ ]:
# ── 실습 과제 ──
# 목표: Step 2. 라벨별 Feature 분포 비교
# 만들 결과: features, colors, subset
# 산출: 그래프 시각화
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
fig, axes = ...
features = ...
colors = ...

for ax, feat in zip(axes, features):
    for label, color in colors.items():
        subset = ...
        ax.hist(subset, bins = ...
    ...
    if feat == 'Flow Duration':
        ...
    ax.legend(fontsize = ...
...
```
</details>


관찰:
- **Flow Duration**: BENIGN 은 넓게 분포, PortScan 은 매우 짧은 쪽에 집중 (한 번 SYN 후 종료).
- **Total Fwd Packets**: DDoS 는 큰 값, PortScan 은 작은 값.
- **SYN Flag Count**: PortScan 은 거의 1에 수렴 (SYN-only).

---
## Step 3. 본문의 '스캔 의심' 규칙 구현

In [ ]:
# ── 실습 과제 ──
# 목표: Step 3. 본문의 '스캔 의심' 규칙 구현
# 만들 결과: rule_mask, df['rule_pred'], y_true, y_pred, tp, fp
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# 규칙: SYN Flag Count > 0.85 그리고 Total Fwd Packets <= 5 그리고 Flow Duration < 500
rule_mask = ...
df['rule_pred'] = ...

# 실제 라벨이 PortScan 인지 여부로 평가
y_true = ...
y_pred = ...

tp = ...
fp = ...
fn = ...
tn = ...

precision = ...
recall = ...
print(f'TP = ...
print(f'Precision = ...
```
</details>


> **공격자 시점**: 룰의 임계값만 알면 우회는 쉽다 — 스캔 속도를 늦추거나 패킷 길이를 조작하면 룰을 빠져나간다.
> 이 한계가 ch11 의 비지도 이상 탐지, ch14 의 지도학습 모델로 이어지는 동기다.

---
## Step 4. 룰 vs 라벨 혼동 행렬을 히트맵으로

In [ ]:
# ── 실습 과제 ──
# 목표: Step 4. 룰 vs 라벨 혼동 행렬을 히트맵으로
# 만들 결과: cm, im
# 산출: 그래프 시각화
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
cm = ...
fig, ax = ...
im = ...
...
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f'{cm.values[i,j]:,}', ha = ...
...
plt.colorbar(im, ax = ...
...
```
</details>


---
## 챌린지

1. DDoS 도 룰로 잡아 보자. 어떤 feature 조합이 유효할까?
2. DDoS 에 대한 precision/recall 을 측정해 보라.

In [ ]:
# ── 실습 과제 ──
# 목표: 챌린지 — 1. DDoS 도 룰로 잡아 보자. 어떤 feature 조합이 유효할까?
# 만들 결과: df['ddos_flag'], hits, tp, fp, fn
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# DDoS 탐지 룰: 대량 순방향 패킷 + 높은 SYN 비율
df['ddos_flag'] = ...
hits = ...
...

# 간이 정밀도/재현율
tp = ...
fp = ...
fn = ...
print(f'정밀도 = ...
```
</details>


---
## 정리

- 룰 기반 탐지는 단순하고 빠르지만 우회되기 쉽다.
- 라벨별 feature 분포 차이가 시각화되면 직관적으로 어떤 feature 가 유효한지 보인다.
- 다음 챕터에서는 실제 PCAP 의 패킷 단위 분석으로 SYN Flood, ARP Spoof, DNS Poison 을 다룬다.